# 用户转向后的部分实收证据封存

In [1]:
from pathlib import Path
import os,json,hashlib
import qlib
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/flow_receipts_20261014';summary=json.loads((card/'summary.json').read_text());plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=str(root/plan['bootstrap_provider']),region='us');receipts=[]
with R.uri_context(summary['recorder_uri']):
 rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_name=card.name)
 for slot in ['baseline','open_a','open_b']:
  part=rec.load_object('inputs/'+slot+'/receipts')
  for receipt in part:
   if receipt.get('path'):assert hashlib.sha256((root/plan['raw_directory']/receipt['path']).read_bytes()).hexdigest()==receipt['sha256']
  receipts.extend(part)
 summary.update(status='interrupted_by_user_redirection',decision='source_collection_stopped_for_training_backtest',captured_requests=len(receipts),cancelled_slots=['recovery_a','recovery_b'],forecast_input_qualified=False,objective_achieved=False,scope='Partial source evidence only. User requested training and backtesting instead of waiting on live data. Original capture interrupted; no future-window requests and no completed-source qualification.')
 rec.save_objects(artifact_path='analysis',interruption=summary,receipts=receipts)
 rec.save_objects(artifact_path='inputs',clock_probe=json.loads((card/'clock_probe.json').read_text()),time_semantics_contract=json.loads((card/'time_semantics_contract.json').read_text()),vendor_definition_notes=(card/'vendor_definition_notes.md').read_text())
 artifacts=Path(summary['recorder_uri'])/summary['experiment_id']/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),captured_requests=len(receipts))
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=summary['experiment_id']);atomic_json(summary,card/'summary.json');atomic_json(receipts,card/'receipts.json');atomic_json({'state':'interrupted_by_user_redirection','captured_requests':len(receipts)},card/'progress.json')
print('Archived partial receipts:',len(receipts))


# 实时采集已停止，任务转回训练和回测

按用户最新要求停止等待实时窗口。原采集进程已收到中断并终止，不再请求10:30窗口。已取得盘前2份及09:45窗口4份回执，保留原始值、请求和接收时间、原严格判定；四份正式窗口响应均在09:46前到达，但含09:46标记的点及超前的last_valid_time，原严格资格仍为false，不解释为已完成分钟信号。

当前notebook只封存已存在的6份回执及辅助时钟/定义证据，不再调用Futu或等待时钟。原采集run的中断状态保留，NAS归档只是部分来源证据，不是完成的研究、预测资格或超额结果。恢复窗口记录为用户取消，不能补成正常采集成功或行情缺失。

下一步使用现有历史数据进行预先固定的训练与完整净费回测。没有修改现役release、没有订单，稳定净超额目标继续保持未完成。
